# AI, economic perceptions, and consumer sentiment

## Reproducible macro baseline and monitoring build

This notebook implements the supplied final-project brief's economic baseline. It does not assert that media bias causes low sentiment. The provided workbook contains macro data only, so the media/causal portions are pre-registered as a data and validation plan rather than filled with invented evidence.

**Research distinction.** A negative sentiment gap is observed sentiment below a macro model prediction. It is an unexplained residual, not proof of irrationality, media bias, exposure, or a cognitive mechanism.


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.pipeline import load_macro_data, load_month_end_vintage_macro_panel, rolling_ols_forecast, model_metrics, build_outputs, corpus_status, validation_status, macro_refresh_status

pd.set_option('display.max_columns', 20)
RAW = ROOT / 'data/raw/Perceptual_Bias_Data.xlsx'
assert RAW.exists(), f'Missing source workbook: {RAW}'
print(f'Project root: {ROOT}')


## 1. Data provenance, timing, and limits

The source workbook supplies the Michigan headline index, U-3 unemployment, PCE YoY, and quarterly real GDP QoQ annualized. The project now also includes historical Current Economic Conditions and Expectations final values transcribed from annual December Michigan reports. Those 2010-2025 subindices are explicitly marked as annual-report backfills rather than original monthly release-vintage observations. The data still do **not** supply respondent-level media exposure or a treatment assignment.

For a monitoring-compatible baseline, inputs are joined only when assumed available: U-3 in the following month, PCE after two month-ends, and GDP one month-end after the quarter. These rules are conservative approximations for this course dataset. A real-time study must replace them with archived release timestamps and vintage values.


In [ ]:
panel = load_macro_data(RAW)
coverage = pd.DataFrame({
    'non_missing': panel[['sentiment','unemployment','pce_yoy','gdp_qoq_annualized']].notna().sum(),
    'first_observation': [panel.loc[panel[c].notna(),'month'].min().date() for c in ['sentiment','unemployment','pce_yoy','gdp_qoq_annualized']],
    'last_observation': [panel.loc[panel[c].notna(),'month'].max().date() for c in ['sentiment','unemployment','pce_yoy','gdp_qoq_annualized']],
})
coverage


## 2. Prespecified economic-only model

The target is the headline Michigan index in month *t*. Features are the latest assumed-available unemployment rate, PCE YoY, and quarterly real GDP growth. We estimate a one-step-ahead rolling OLS forecast using only earlier rows, a 120-month window, and 60-month minimum training sample. The 95% interval uses the in-window residual standard deviation. It is a simple benchmark, not a structural model.

This design keeps GDP quarterly rather than fabricating monthly values. The availability join carries the latest released GDP reading forward only after its assumed release date.


In [ ]:
analysis = panel.loc[panel['month'] >= '2010-01-31'].copy()
baseline_features = ['unemployment', 'pce_yoy', 'gdp_qoq_annualized']
baseline = rolling_ols_forecast(analysis, baseline_features)
metrics = model_metrics(baseline)
metrics


In [ ]:
baseline.dropna(subset=['prediction']).tail(12)[['month','sentiment','prediction','prediction_lower_95','prediction_upper_95','sentiment_gap','training_n']]


In [ ]:
oos = baseline.dropna(subset=['prediction']).copy()
print('Share inside nominal 95% interval:', f"{oos['inside_95_interval'].mean():.1%}")
print('Largest negative gaps (unexplained pessimism under this baseline):')
oos.nsmallest(10, 'sentiment_gap')[['month','sentiment','prediction','sentiment_gap']]


## 3. Autocorrelation and genuine month-end vintages

A lagged-sentiment sensitivity model explicitly accounts for serial dependence; it is a forecasting comparator, not a structural interpretation. The ALFRED panel uses initial-release UNRATE, PCEPI, and GDPC1 values known by each calendar month-end, rather than the baseline's assumed lags. It still cannot make an exact survey-release-time claim because historical Michigan release timestamps are incomplete.


In [ ]:
dynamic = analysis.copy()
dynamic['sentiment_lag1'] = dynamic['sentiment'].shift(1)
dynamic_model = rolling_ols_forecast(dynamic, baseline_features + ['sentiment_lag1'])
vintage_panel = load_month_end_vintage_macro_panel(RAW)
vintage_features = ['unemployment_vintage', 'pce_yoy_vintage', 'gdp_qoq_annualized_vintage']
vintage_model = rolling_ols_forecast(vintage_panel.dropna(subset=vintage_features), vintage_features) if vintage_panel is not None else None
pd.DataFrame([
    {'model':'Economic only', **model_metrics(baseline)},
    {'model':'Economic + lagged sentiment', **model_metrics(dynamic_model)},
    {'model':'Month-end ALFRED initial vintages', **({} if vintage_model is None else model_metrics(vintage_model))},
])


## 4. Supplementary extension: household-stress proxies

The brief asks whether omitted economic pressures could explain the gap. With the supplied data, we can test two narrow proxies: the 12-month change in U-3 and a negative-GDP recession signal. This is not a measure of household income, price levels, affordability, debt burden, or lived experience. The extension is assessed only by its held-out prediction metrics; coefficient stories are deliberately avoided here.


In [ ]:
extension_features = baseline_features + ['unemployment_12m_change', 'recession_signal']
extension = rolling_ols_forecast(analysis, extension_features)
comparison = pd.DataFrame([
    {'model':'Economic only', **model_metrics(baseline)},
    {'model':'Economic + stress proxies', **model_metrics(extension)},
])
comparison


## 5. Expanded public controls and real-time guardrail

Run `python scripts/refresh_public_macro.py` before this section. The refresh obtains current-vintage public snapshots with hashes and retrieval metadata; it does not make a historical vintage claim. The registry also records direct/licensed inputs still needed for Michigan current conditions and expectations. Housing affordability is monitored but excluded from the long rolling specification when coverage is insufficient.


In [ ]:
refresh = macro_refresh_status()
controls_path = ROOT / 'data/external/processed/public_macro_controls.csv'
print(json.dumps(refresh, indent=2))
if controls_path.exists():
    controls = pd.read_csv(controls_path)
    print('Available control columns:', ', '.join(controls.columns))
    display(controls.tail())
else:
    print('No refresh found. Run scripts/refresh_public_macro.py.')


## 5. Media-measurement validation and model gate

The source-blinded all-field validation contains 312 paired records and covers tone, salience, framing, factual calibration, and political attribution. Its 60-item holdout remains reserved for a final coding-model claim. The availability-dated historical archive now supplies 89 aligned months, exceeding the 84-month threshold for the pre-specified chronological media comparison. Treat its forecast comparison as associational rather than causal; inspect the generated media-model status for the current gate and diagnostics.


In [ ]:
print(json.dumps(corpus_status(), indent=2))
print(json.dumps(validation_status(), indent=2))


## 6. Observational association and causal-design boundary

Run `python scripts/build_causal_association_analysis.py` before this section. The implemented supplement estimates lagged associations between independently validated historical negative tone and three Michigan outcomes, conditional on outcome persistence, macro controls, calendar seasonality, and HAC uncertainty. It also reports a future-media placebo. This is **not** an identified causal estimate: the data have no respondent exposure, random assignment, or credible exogenous instrument. The 2010-2025 subindices are final-value annual backfills and are used only retrospectively.

`docs/causal_design_protocol.md` specifies the stronger next step: a consented, randomized truthful-framing experiment. Its effect would apply to the recruited sample, not automatically to the national Michigan index.


In [ ]:
causal_status_path = ROOT / 'data/processed/causal_analysis_status.json'
causal_results_path = ROOT / 'data/processed/causal_association_results.csv'
if causal_status_path.exists() and causal_results_path.exists():
    causal_status = json.loads(causal_status_path.read_text())
    print(json.dumps({k: causal_status[k] for k in ['claim', 'observations', 'coverage', 'placebo_test', 'not_identified']}, indent=2))
    display(pd.read_csv(causal_results_path))
else:
    print('Run scripts/build_causal_association_analysis.py first.')


## 7. Historical media comparison and input contracts

Run `python scripts/build_media_model_analysis.py` before rebuilding the pipeline. Its primary specification uses a one-month media lag, calendar-seasonality terms, chronological holdouts, residual diagnostics, coefficient-stability summaries, and a contemporaneous-media sensitivity check. It requires 84 aligned, availability-dated months; with fewer rows it emits an explicit blocked status.

The dashboard also checks contracts for Michigan Current Conditions/Expectations, release-vintage macro observations, survey subgroups, and verified release/events. Populate the matching files in `data/templates/` from authoritative sources before treating those requirements as complete.


In [ ]:
corpus = corpus_status()
print(json.dumps(corpus, indent=2))
media_path = ROOT / 'data/media_monthly.csv'
if media_path.exists():
    media = pd.read_csv(media_path, parse_dates=['month'])
    print('Provisional media aggregate found:', media.shape)
    display(media)
else:
    print('No monthly media measures yet; no media effect is estimated.')


## 5. Causal design and falsification checks

**Feasible present claim:** the macro and any future media features are associated with sentiment/prediction errors under the stated timing assumptions.

**Not identified:** audience exposure, change in individual beliefs, and causal effect on the national Michigan index.

**Preferred next design:** randomized, truthful framing experiment with balanced factual summaries, pre-registered outcomes, consent, and review approval. It estimates a sample framing effect, not an effect on the Michigan index without representative sampling, exposure estimates, weighting, and transportability assumptions.

**Falsifications:** (a) media features fail to improve untouched forecasts, (b) effects disappear when sources are balanced or exact releases are compared, (c) swapped political names/source labels change the classifier but not the text, (d) placebo/later features appear to predict earlier sentiment, or (e) the gap is absorbed by documented economic/household controls.


## 6. Produce reproducible dashboard inputs

The next cell writes the normalized panel, both out-of-sample results, model summary, and a small JSON payload. The dashboard uses that payload and preserves missing media measures as missing.


In [ ]:
payload = build_outputs(RAW)
print(json.dumps({'as_of': payload['metadata']['as_of'], 'metrics': payload['metrics']}, indent=2))
print('Dashboard payload:', ROOT / 'data/processed/monitoring_payload.json')


## Interpretation

The model is intentionally modest. A low sentiment reading outside the economic-only forecast band raises a measurement question. It does not distinguish omitted household pressures, survey timing, macro revisions, selective information, framing, reverse causality, or cognitive bias. The next evidence needed is a documented corpus plus exposure-aware design, not stronger language about the residual.
